## Usage Guide

**Best ways to interact with this chatbot:**

### 1. **Single Query Cell** (Recommended for focused questions)
- Scroll to the "Interactive Chat" section
- Edit the `query` variable with your question
- Run the cell to get a response with sources

### 2. **Multi-Query Testing** (Good for batch testing)
- Use the "Quick Multi-Query Testing" section
- Add multiple queries to the list
- Run once to get all responses

### 3. **Retrieval Analysis** (Debug/tune retrieval)
- Use the "Retrieval Analysis" section
- See exactly what documents are being retrieved
- Useful for understanding why you got certain responses

### 4. **CLI Mode** (For interactive conversation)
- Exit the notebook
- Run: `python gemini_rag_chatbot.py`
- This gives you a true multi-turn chat interface

**Configuration Tips:**
- Modify `RAG_TOP_K` in `.env` to retrieve more/fewer documents
- Adjust `RAG_SCORE_THRESHOLD` to make retrieval more/less strict
- Change `TEMPERATURE` to make responses more deterministic (0.0) or creative (2.0)

In [ ]:
query = "communication tips"

print(f"Query: '{query}'")
print(f"Top K: {rag.ChatbotConfig.TOP_K}")
print(f"Score Threshold: {rag.ChatbotConfig.SCORE_THRESHOLD}\n")

docs = retriever.get_relevant_documents(query)

print(f"Retrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.page_content}")
    print("-" * 70)

## Retrieval Analysis

See what documents are being retrieved for a query:

In [ ]:
queries = [
    "What are some good conversation starters?",
    "How can I improve communication with my partner?",
    "What are some romantic ideas?",
]

for i, q in enumerate(queries, 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {q}")
    print('='*70)
    
    try:
        response = chain.invoke(q)
        print(f"\nResponse:\n{response}")
    except Exception as e:
        print(f"Error: {e}")

## Quick Multi-Query Testing

Run multiple queries in one cell. Edit the queries list below:

# Gemini RAG Chatbot - Full Test Suite

This notebook tests all major components of the Gemini RAG chatbot without requiring interactive input.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import gemini_rag_chatbot as rag

print("✓ Imports successful")

## TEST 1: Environment Variable Loading

In [ ]:
print("\n" + "=" * 70)
print("TEST 1: Environment Variable Loading")
print("=" * 70)

env_path = rag.ChatbotConfig.PROJECT_DIR / ".env"
print(f"Loading .env from: {env_path}")
print(f"File exists: {env_path.exists()}")

load_dotenv(dotenv_path=env_path)

api_key = os.getenv("GOOGLE_API_KEY")
print(f"✓ GOOGLE_API_KEY loaded: {'***' + api_key[-4:] if api_key else 'NOT FOUND'}")

gemini_model = os.getenv("GEMINI_MODEL")
print(f"✓ GEMINI_MODEL: {gemini_model}")

top_k = os.getenv("RAG_TOP_K")
print(f"✓ RAG_TOP_K: {top_k}")

if not api_key:
    print("❌ GOOGLE_API_KEY not found!")
    raise ValueError("Missing GOOGLE_API_KEY")

print("✅ Environment loading PASSED")

## TEST 2: Embeddings CSV Loading

In [ ]:
print("\n" + "=" * 70)
print("TEST 2: Embeddings CSV Loading")
print("=" * 70)

csv_path = rag.ChatbotConfig.EMBEDDINGS_CSV
abs_path = rag.ChatbotConfig.PROJECT_DIR / csv_path
print(f"Loading from: {abs_path}")
print(f"File exists: {abs_path.exists()}")

try:
    texts, embeddings = rag.load_embeddings_from_csv(csv_path)
    print(f"✓ Loaded {len(texts)} documents")
    print(f"✓ Embeddings shape: {embeddings.shape}")
    print(f"✓ First text (truncated): {texts[0][:100]}...")
    print(f"✓ First embedding (first 5 dims): {embeddings[0][:5]}")
    print("✅ Embeddings loading PASSED")
except Exception as e:
    print(f"❌ Failed to load embeddings: {e}")
    raise

## TEST 3: Vector Store Creation

In [ ]:
print("\n" + "=" * 70)
print("TEST 3: Vector Store Creation")
print("=" * 70)

try:
    vector_store = rag.create_vector_store(texts, embeddings)
    print(f"✓ Vector store created")
    print(f"✓ FAISS index entries: {vector_store.index.ntotal}")
    print(f"✓ Index dimension: {vector_store.index.d}")
    print("✅ Vector store creation PASSED")
except Exception as e:
    print(f"❌ Failed to create vector store: {e}")
    raise

## TEST 4: LLM Initialization

In [ ]:
print("\n" + "=" * 70)
print("TEST 4: LLM Initialization")
print("=" * 70)

try:
    llm = rag.create_gemini_llm()
    print(f"✓ LLM initialized: {rag.ChatbotConfig.GEMINI_MODEL}")
    print(f"✓ Temperature: {rag.ChatbotConfig.TEMPERATURE}")
    print(f"✓ Max tokens: {rag.ChatbotConfig.MAX_TOKENS}")
    print("✅ LLM initialization PASSED")
except ValueError as e:
    print(f"❌ Configuration error: {e}")
    raise
except Exception as e:
    print(f"❌ Failed to initialize LLM: {e}")
    raise

## TEST 5: RAG Chain Assembly

In [ ]:
print("\n" + "=" * 70)
print("TEST 5: RAG Chain Assembly")
print("=" * 70)

try:
    chain, retriever = rag.create_rag_chain(vector_store, llm)
    print(f"✓ RAG chain created")
    print(f"✓ Retriever configured with top_k={rag.ChatbotConfig.TOP_K}")
    print(f"✓ Score threshold: {rag.ChatbotConfig.SCORE_THRESHOLD}")
    print("✅ RAG chain assembly PASSED")
except Exception as e:
    print(f"❌ Failed to assemble RAG chain: {e}")
    raise

## TEST 6: Retrieval Functionality

In [ ]:
print("\n" + "=" * 70)
print("TEST 6: Retrieval Functionality")
print("=" * 70)

try:
    # Test with a simple query
    test_query = "question"
    docs = retriever.get_relevant_documents(test_query)
    print(f"✓ Retrieved {len(docs)} documents for query: '{test_query}'")
    if docs:
        print(f"✓ Top result (truncated): {docs[0].page_content[:100]}...")
    print("✅ Retrieval PASSED")
except Exception as e:
    print(f"❌ Retrieval failed: {e}")
    raise

## TEST 7: Simple Query Test

In [ ]:
print("\n" + "=" * 70)
print("TEST 7: Simple Query Test (Using RAG Chain)")
print("=" * 70)

test_query = "What is the most common question?"
print(f"\nQuery: {test_query}")
print("\nGenerating response...")

try:
    response = chain.invoke(test_query)
    print(f"\n✓ Response generated successfully")
    print(f"\nResponse:\n{response}")
    print("\n✅ Query test PASSED")
except Exception as e:
    print(f"❌ Query failed: {e}")
    raise

## TEST SUMMARY

In [ ]:
print("\n" + "=" * 70)
print("✅ ALL TESTS PASSED!")
print("=" * 70)
print("\n🎉 Chatbot is ready to use!")
print("\nTo run the interactive chatbot, use:")
print("  python gemini_rag_chatbot.py")
print("\n" + "=" * 70)

In [ ]:
# Edit this query and run the cell
query = "What are some creative ideas for intimacy?"

print(f"🤔 Query: {query}")
print("\n⏳ Retrieving context and generating response...\n")

# Get relevant documents
docs = retriever.get_relevant_documents(query)
print(f"📚 Retrieved {len(docs)} relevant documents\n")

# Generate response
response = chain.invoke(query)

print("🤖 Response:")
print("-" * 70)
print(response)
print("-" * 70)

# Show sources
if docs:
    print("\n📖 Sources Used:")
    for i, doc in enumerate(docs[:3], 1):
        excerpt = doc.page_content[:150].replace("\n", " ")
        print(f"\n{i}. {excerpt}...")

## Interactive Chat - Ask Your Questions

You can now query the chatbot! Modify the query below and run the cell to get a response.